# PepSeqPred HPC API Smoke Test

This notebook validates the pretrained and artifact-based inference API on a GPU-enabled HPC environment.

## One-time terminal setup on HPC

Run these commands from your repo checkout before opening the notebook:

```bash
git checkout feat/pretrained-api
git pull
conda activate pepseqpred
python -m pip install --upgrade pip
python -m pip install -e .[dev]
jupyter lab
```

If this branch has already been merged and tagged, you can replace editable install with `pip install -U pepseqpred`.

In [ ]:
from importlib.resources import as_file, files
from pathlib import Path
import platform
import torch
import pandas as pd

import pepseqpred
from pepseqpred import (
    PepSeqPredictor,
    list_pretrained_models,
    load_pretrained_predictor,
    load_predictor,
    predict_sequence,
    predict_fasta,
)

In [ ]:
print("pepseqpred version:", pepseqpred.__version__)
print("torch version:", torch.__version__)
print("python:", platform.python_version())
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Run this notebook on a GPU node.")

print("cuda device count:", torch.cuda.device_count())
print("cuda[0]:", torch.cuda.get_device_name(0))

DEVICE = "cuda"

In [ ]:
catalog = list_pretrained_models()
catalog_df = pd.DataFrame([
    {
        "model_id": m.model_id,
        "aliases": ", ".join(m.aliases),
        "expected_esm_model": m.expected_esm_model,
        "n_members": m.n_members,
        "is_default": m.is_default,
    }
    for m in catalog
])
catalog_df

In [ ]:
predictor_default = load_pretrained_predictor(device=DEVICE, k_folds=1)
predictor_alias = load_pretrained_predictor("flagship1", device=DEVICE, k_folds=1)
predictor_cls = PepSeqPredictor.from_pretrained("flagship2-v1", device=DEVICE, k_folds=1)

print("default:", predictor_default.pretrained_meta.get("model_id"), "members:", predictor_default.n_members)
print("alias:", predictor_alias.pretrained_meta.get("model_id"), "members:", predictor_alias.n_members)
print("classmethod:", predictor_cls.pretrained_meta.get("model_id"), "members:", predictor_cls.n_members)

In [ ]:
sequence = "MKWVTFISLLFLFSSAYSRGVFRRDTHKSEIAHRFKDLGE"
result = predictor_default.predict_sequence(sequence, header="toy_seq")

print("header:", result.header)
print("length:", result.length)
print("n_epitopes:", result.n_epitopes)
print("binary_mask prefix:", result.binary_mask[:25])
print("pretrained id:", result.meta.get("pretrained", {}).get("model_id"))

assert result.length == len(result.sequence)
assert len(result.binary_mask) == result.length
assert result.meta.get("pretrained", {}).get("model_id") == "flagship2-v1"

In [ ]:
workdir = Path("localdata/tmp_api_hpc_smoke")
workdir.mkdir(parents=True, exist_ok=True)

fasta_input = workdir / "smoke_input.fasta"
fasta_input.write_text(
    ">seq1\nACDEFGHIKLMNPQ\n>seq2\nMNPQRSTVWYACDEFG\n",
    encoding="utf-8",
)

method_output = workdir / "method_masks.fasta"
wrapper_output = workdir / "wrapper_masks.fasta"

method_results = predictor_default.write_fasta_predictions(
    fasta_input=fasta_input,
    output_fasta=method_output,
)

manifest_ref = files("pepseqpred.api").joinpath(
    "pretrained_artifacts/flagship2-v1/ensemble_manifest.json"
)
with as_file(manifest_ref) as manifest_path:
    artifact_predictor = load_predictor(
        model_artifact=manifest_path,
        model_name="esm2_t33_650M_UR50D",
        device=DEVICE,
        k_folds=1,
    )
    wrapper_single = predict_sequence(
        model_artifact=manifest_path,
        protein_seq=sequence,
        header="wrapper_seq",
        model_name="esm2_t33_650M_UR50D",
        device=DEVICE,
        k_folds=1,
    )
    wrapper_fasta = predict_fasta(
        model_artifact=manifest_path,
        fasta_input=fasta_input,
        output_fasta=wrapper_output,
        model_name="esm2_t33_650M_UR50D",
        device=DEVICE,
        k_folds=1,
    )

assert len(method_results) == 2
assert len(wrapper_fasta) == 2
assert method_output.exists()
assert wrapper_output.exists()
assert artifact_predictor.n_members == 1
assert wrapper_single.length == len(sequence)

print("All API smoke checks passed.")
print("Artifacts:", workdir.resolve())